# Phase 2: EDA and preprocessing

Loads the three yearly tables, adds stable cell and block ids (D-022), the within-year LST
anomaly target (D-021) and the urban flag (D-019), and writes `data/processed/uhi_pune_clean.parquet`.
The logic is in `src/mirageai/preprocess.py`; `scripts/pipeline/run_phase2.py` runs it end to end.

In [1]:
import pandas as pd
from mirageai import config, results as R

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
from mirageai import preprocess

## Clean table, missing values and collinearity

Needs `data/` (not shipped; see README → Data).

In [2]:
if config.CLEAN_TABLE.exists():
    df = preprocess.load_clean()
    print(f"{len(df):,} rows x {df.shape[1]} columns, {int(df.isna().sum().sum())} nulls")
    display(preprocess.missing_report(df))
    for dropped in ("frac_built", "frac_bare"):
        vif = preprocess.vif_table(df, [f for f in config.FEATURES if f != dropped])
        print(f"VIF without {dropped}: median {vif.VIF.median():.1f}")  # D-023
else:
    print("data/processed/uhi_pune_clean.parquet not found: run scripts/pipeline/run_phase2.py")

245,949 rows x 28 columns, 0 nulls


,rows,null_cells,below_min_obs,min_n_obs,median_n_obs
year,,,,,
2023,81983,0,0,4.35,7.0
2024,81983,0,0,7.82,10.0
2025,81983,0,0,7.00,9.0


VIF without frac_built: median 2.2


VIF without frac_bare: median 11.8


## Correlation with the target, all cells vs urban cells

NDBI is the strongest correlate and tracks **dry soil**, not buildings (D-024).

In [3]:
R.load_table("phase2_lst_anom_correlations").rename(columns={"Unnamed: 0": "feature"})

,feature,all_cells,urban_only,difference
0,NDBI,0.635,0.659,0.024
1,frac_grass,0.593,0.389,-0.203
2,frac_tree,-0.491,-0.455,0.036
3,NDVI,-0.436,-0.547,-0.111
4,frac_water,-0.364,-0.045,0.320
5,MNDWI,-0.283,0.069,0.352
6,elevation,0.259,0.103,-0.156
7,dist_water_m,0.194,0.018,-0.175
8,frac_built,-0.192,0.009,0.200
9,slope,0.127,0.013,-0.114


![Target distributions](../outputs/figures/02_target_distributions.png)

![Correlation heatmap](../outputs/figures/02_correlation_heatmap.png)